# Iteration 06 - 5% More Admissions Validation

## Aim

Extend validation to the published scenario of 5% more admissions and present the simulation results in a table shaped like Table 2 in Monks et al.


## Prompt Used

In iteration 6, extend validation to the published scenario of 5% more admissions. Present the results of the simulation in table form, similar to Table 2 in Monks et al.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from stroke_sim.config import SimulationSettings
from stroke_sim.runner import run_replications
from stroke_sim.validation import (
    PUBLISHED_CURRENT_ADMISSIONS_ACUTE,
    PUBLISHED_CURRENT_ADMISSIONS_REHAB,
    PUBLISHED_MORE_ADMISSIONS_ACUTE,
    PUBLISHED_MORE_ADMISSIONS_REHAB,
    build_table_2_style,
    compare_to_published_table,
)


## Scenario Runs

We compare two scenarios:

- current admissions: arrival multiplier = 1.00
- 5% more admissions: arrival multiplier = 1.05


In [ ]:
base_kwargs = dict(run_length_days=365 * 5, warm_up_days=365 * 3, replications=30)
current_audit = run_replications(settings=SimulationSettings(arrival_rate_multiplier=1.0, **base_kwargs))
more_audit = run_replications(settings=SimulationSettings(arrival_rate_multiplier=1.05, **base_kwargs))


## Acute Comparison Table


In [ ]:
acute_current = compare_to_published_table(
    current_audit,
    column='acute_occupancy',
    published=PUBLISHED_CURRENT_ADMISSIONS_ACUTE,
)
acute_more = compare_to_published_table(
    more_audit,
    column='acute_occupancy',
    published=PUBLISHED_MORE_ADMISSIONS_ACUTE,
)
acute_table_2 = build_table_2_style(
    acute_current,
    acute_more,
    bed_label='No. acute beds',
    include_only_more_beds={10, 11, 12, 13, 14},
)
acute_table_2


## Rehab Comparison Table


In [ ]:
rehab_current = compare_to_published_table(
    current_audit,
    column='rehab_occupancy',
    published=PUBLISHED_CURRENT_ADMISSIONS_REHAB,
)
rehab_more = compare_to_published_table(
    more_audit,
    column='rehab_occupancy',
    published=PUBLISHED_MORE_ADMISSIONS_REHAB,
)
rehab_table_2 = build_table_2_style(
    rehab_current,
    rehab_more,
    bed_label='No. rehab beds',
    include_only_more_beds={12, 13, 14, 15, 16},
)
rehab_table_2


## Combined Table 2 Style Output

This combines the acute and rehab sections into a single presentation table similar to Table 2 in the paper.


In [ ]:
import pandas as pd

table_2_style = pd.concat([acute_table_2, rehab_table_2], ignore_index=True)
table_2_style


## Short Summary


In [ ]:
summary = {
    'acute_current_mean_abs_error': round(float(acute_current['abs_error'].mean()), 4),
    'acute_more_mean_abs_error': round(float(acute_more['abs_error'].mean()), 4),
    'rehab_current_mean_abs_error': round(float(rehab_current['abs_error'].mean()), 4),
    'rehab_more_mean_abs_error': round(float(rehab_more['abs_error'].mean()), 4),
}
summary


## Tester Notes

- The 5% more admissions scenario is implemented by multiplying all arrival rates by 1.05, equivalent to dividing the mean inter-arrival times by 1.05.
- Delay estimates continue to use the Erlang-loss-style occupancy calculation that aligned best with the published base-case tables.
- The table is presentation-oriented and shaped to resemble Monks et al. Table 2.
